# 23 curated enrichment

**Purpose:** Run curated over-representation analyses.

**Original analysis notebook:** `22_functional_enrichment_pathway_interpretation_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 41 — Functional enrichment and pathway interpretation (HVG8000 r0.6)

**Goal:** convert the large gene-level DE outputs into interpretable biological pathways/programs.

This notebook summarizes enrichment for:
1. malignant-state pseudobulk DE markers,
2. condition-specific malignant pseudobulk DE contrasts,
3. within-state condition-specific DE,
4. malignant program-score differences across conditions.

The main analysis is an **offline over-representation analysis (ORA)** using curated glioma/scRNA-relevant gene sets. If external `.gmt` gene-set files are found, they are also loaded automatically.


In [ ]:

import os
import re
import gzip
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = PROJECT_DIR / "figures" / "functional_enrichment_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

TAG = "HVG8000_r0_6"
OUT_PREFIX = f"functional_enrichment_{TAG}"

# Inputs from previous steps
STATE_SIG_TSV = METADATA_DIR / f"malignant_state_pseudobulk_DE_state_vs_rest_significant_{TAG}.tsv"
STATE_TOP_TSV = METADATA_DIR / f"malignant_state_pseudobulk_DE_state_vs_rest_top100_up_down_{TAG}.tsv"

COND_OVERALL_SIG_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv"
COND_OVERALL_TOP_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_overall_malignant_pairwise_DE_top100_up_down.tsv"
WITHIN_STATE_TOP_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_within_state_condition_DE_top50_up_down.tsv"
WITHIN_STATE_SUMMARY_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_within_state_condition_DE_test_summary.tsv"

PROGRAM_GLOBAL_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_program_score_global_kruskal.tsv"
PROGRAM_PAIRWISE_TSV = METADATA_DIR / f"condition_malignant_pseudobulk_DE_{TAG}_program_score_pairwise_mannwhitney.tsv"

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def read_tsv_optional(path):
    path = Path(path)
    if not path.exists():
        print(f"[MISSING] {path}")
        return pd.DataFrame()
    print(f"[READ] {path}")
    return pd.read_csv(path, sep="\t", low_memory=False)

readiness = []
for label, path in [
    ("state_significant", STATE_SIG_TSV),
    ("state_top", STATE_TOP_TSV),
    ("condition_overall_significant", COND_OVERALL_SIG_TSV),
    ("condition_overall_top", COND_OVERALL_TOP_TSV),
    ("within_state_top", WITHIN_STATE_TOP_TSV),
    ("within_state_summary", WITHIN_STATE_SUMMARY_TSV),
    ("program_global", PROGRAM_GLOBAL_TSV),
    ("program_pairwise", PROGRAM_PAIRWISE_TSV),
]:
    readiness.append({
        "input": label,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan
    })

readiness_df = pd.DataFrame(readiness)
write_tsv_replace(readiness_df, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness_df)


In [ ]:

# -------------------------
# Helper functions
# -------------------------

def first_existing_col(df, candidates):
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c in df.columns:
            return c
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

def infer_gene_col(df):
    return first_existing_col(df, [
        "gene", "gene_symbol", "symbol", "names", "index", "var_names", "features", "Gene", "GeneID"
    ])

def infer_lfc_col(df):
    return first_existing_col(df, [
        "log2fc", "log2FC", "logFC", "avg_log2FC", "lfc", "coef", "estimate"
    ])

def infer_q_col(df):
    return first_existing_col(df, [
        "q_value", "qvalue", "qval", "padj", "p_adj", "p_adj_bh", "FDR", "fdr", "adj_pval", "p_adjusted"
    ])

def infer_p_col(df):
    return first_existing_col(df, [
        "p_value", "pvalue", "pval", "p", "P.Value"
    ])

def infer_group_cols(df):
    candidates = [
        "condition_malignant_state",
        "malignant_state",
        "state",
        "contrast",
        "comparison",
        "test",
        "group",
        "cluster",
        "final_annotation_refined",
        "program"
    ]
    return [c for c in candidates if c in df.columns]

def standardize_de_table(df, label="de_table"):
    if df is None or df.empty:
        return pd.DataFrame(), {
            "label": label, "n_rows": 0, "gene_col": None, "lfc_col": None, "q_col": None, "p_col": None,
            "group_cols": ""
        }
    df = df.copy()
    gene_col = infer_gene_col(df)
    lfc_col = infer_lfc_col(df)
    q_col = infer_q_col(df)
    p_col = infer_p_col(df)
    group_cols = infer_group_cols(df)

    if gene_col is None:
        raise ValueError(f"No gene column found for {label}. Columns: {list(df.columns)}")

    df["gene_std"] = df[gene_col].astype(str)
    if lfc_col is not None:
        df["lfc_std"] = pd.to_numeric(df[lfc_col], errors="coerce")
    else:
        df["lfc_std"] = np.nan
    if q_col is not None:
        df["q_std"] = pd.to_numeric(df[q_col], errors="coerce")
    else:
        df["q_std"] = np.nan
    if p_col is not None:
        df["p_std"] = pd.to_numeric(df[p_col], errors="coerce")
    else:
        df["p_std"] = np.nan

    # Make a robust group label.
    if group_cols:
        df["group_std"] = df[group_cols].astype(str).agg("__".join, axis=1)
    else:
        df["group_std"] = label

    # Direction of effect.
    df["direction_std"] = np.where(df["lfc_std"] >= 0, "up", "down")
    df.loc[df["lfc_std"].isna(), "direction_std"] = "unknown"

    info = {
        "label": label,
        "n_rows": len(df),
        "gene_col": gene_col,
        "lfc_col": lfc_col,
        "q_col": q_col,
        "p_col": p_col,
        "group_cols": ",".join(group_cols)
    }
    return df, info

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    ok = np.isfinite(pvals)
    q = np.full(len(pvals), np.nan)
    if ok.sum() > 0:
        q[ok] = multipletests(pvals[ok], method="fdr_bh")[1]
    return q

def parse_gmt(gmt_path):
    gene_sets = {}
    with open(gmt_path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                name = parts[0]
                genes = [g.strip().upper() for g in parts[2:] if g.strip()]
                gene_sets[name] = sorted(set(genes))
    return gene_sets

def collect_external_gmts():
    candidates = []
    for base in [
        PROJECT_DIR / "resources",
        PROJECT_DIR / "resources" / "gene_sets",
        PROJECT_DIR / "metadata",
        PROJECT_DIR / "gene_sets",
    ]:
        if base.exists():
            candidates.extend(sorted(base.glob("*.gmt")))
            candidates.extend(sorted(base.glob("**/*.gmt")))
    return sorted(set(candidates))

def clean_gene_set_dict(gene_sets):
    out = {}
    for k, genes in gene_sets.items():
        gs = sorted(set([str(g).upper() for g in genes if pd.notna(g) and str(g).strip()]))
        if len(gs) >= 3:
            out[str(k)] = gs
    return out

def run_ora(gene_lists, gene_sets, universe, min_overlap=2):
    universe = set([str(g).upper() for g in universe if pd.notna(g)])
    results = []
    for gene_list_name, genes in gene_lists.items():
        selected = set([str(g).upper() for g in genes if pd.notna(g)])
        selected = selected & universe
        n_selected = len(selected)
        for set_name, set_genes in gene_sets.items():
            gs = set(set_genes) & universe
            if len(gs) == 0 or n_selected == 0:
                continue
            overlap = selected & gs
            k = len(overlap)
            if k < min_overlap:
                continue

            # 2x2 Fisher exact table:
            # selected in set, selected not in set, not selected in set, not selected not in set
            a = k
            b = n_selected - k
            c = len(gs) - k
            d = len(universe) - a - b - c
            if min(a, b, c, d) < 0:
                continue
            odds, p = fisher_exact([[a, b], [c, d]], alternative="greater")
            results.append({
                "gene_list": gene_list_name,
                "gene_set": set_name,
                "overlap_n": a,
                "selected_n": n_selected,
                "gene_set_n": len(gs),
                "universe_n": len(universe),
                "odds_ratio": odds,
                "p_value": p,
                "overlap_genes": ",".join(sorted(overlap))
            })
    res = pd.DataFrame(results)
    if len(res) > 0:
        res["q_value"] = bh_fdr(res["p_value"].values)
        res["neglog10_q"] = -np.log10(np.clip(res["q_value"].astype(float), 1e-300, 1.0))
        res = res.sort_values(["q_value", "p_value", "odds_ratio"], ascending=[True, True, False]).reset_index(drop=True)
    return res

def make_gene_lists_from_de(df, label_prefix, q_cut=0.05, abs_lfc_cut=0.5, use_if_no_q=True, top_n_if_no_q=100):
    gene_lists = {}
    if df is None or df.empty:
        return gene_lists

    d = df.copy()
    # If q exists, use significance threshold.
    has_q = d["q_std"].notna().any()
    has_lfc = d["lfc_std"].notna().any()

    for group, sub in d.groupby("group_std", dropna=False):
        sub = sub.copy()
        if has_q and has_lfc:
            up = sub[(sub["q_std"] <= q_cut) & (sub["lfc_std"] >= abs_lfc_cut)]["gene_std"].dropna().astype(str).tolist()
            down = sub[(sub["q_std"] <= q_cut) & (sub["lfc_std"] <= -abs_lfc_cut)]["gene_std"].dropna().astype(str).tolist()
        elif has_lfc:
            sub2 = sub.sort_values("lfc_std", ascending=False)
            up = sub2.head(top_n_if_no_q)["gene_std"].dropna().astype(str).tolist()
            down = sub2.tail(top_n_if_no_q)["gene_std"].dropna().astype(str).tolist()
        else:
            # fallback: all genes as an undirected list
            up = sub["gene_std"].dropna().astype(str).tolist()
            down = []

        if len(up) >= 3:
            gene_lists[f"{label_prefix}__{group}__up"] = up
        if len(down) >= 3:
            gene_lists[f"{label_prefix}__{group}__down"] = down

    return gene_lists

def plot_enrichment_heatmap(enrich_df, out_png, title, top_gene_sets=25, top_lists=30, value_col="neglog10_q"):
    if enrich_df is None or enrich_df.empty:
        return None

    d = enrich_df.copy()
    # Keep strongest pairs.
    d = d.sort_values("q_value").head(500)
    top_sets = d.groupby("gene_set")[value_col].max().sort_values(ascending=False).head(top_gene_sets).index.tolist()
    top_lists_idx = d.groupby("gene_list")[value_col].max().sort_values(ascending=False).head(top_lists).index.tolist()
    d = d[d["gene_set"].isin(top_sets) & d["gene_list"].isin(top_lists_idx)]
    if d.empty:
        return None

    mat = d.pivot_table(index="gene_set", columns="gene_list", values=value_col, aggfunc="max", fill_value=0.0)
    # order rows/cols by max signal
    mat = mat.loc[mat.max(axis=1).sort_values(ascending=False).index]
    mat = mat[mat.max(axis=0).sort_values(ascending=False).index]

    fig_w = max(8, 0.32 * mat.shape[1] + 4)
    fig_h = max(6, 0.32 * mat.shape[0] + 2)
    plt.figure(figsize=(fig_w, fig_h))
    im = plt.imshow(mat.values, aspect="auto")
    plt.colorbar(im, label="-log10(FDR)")
    plt.xticks(range(mat.shape[1]), mat.columns, rotation=90, fontsize=7)
    plt.yticks(range(mat.shape[0]), mat.index, fontsize=8)
    plt.title(title)
    plt.tight_layout()
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"[SAVED FIG] {out_png}")
    return out_png


In [ ]:

# -------------------------
# Load DE and program-score outputs
# -------------------------

state_sig_raw = read_tsv_optional(STATE_SIG_TSV)
state_top_raw = read_tsv_optional(STATE_TOP_TSV)
cond_overall_sig_raw = read_tsv_optional(COND_OVERALL_SIG_TSV)
cond_overall_top_raw = read_tsv_optional(COND_OVERALL_TOP_TSV)
within_state_top_raw = read_tsv_optional(WITHIN_STATE_TOP_TSV)
program_global = read_tsv_optional(PROGRAM_GLOBAL_TSV)
program_pairwise = read_tsv_optional(PROGRAM_PAIRWISE_TSV)

std_tables = []
state_sig, info = standardize_de_table(state_sig_raw, "malignant_state_significant")
std_tables.append(info)
state_top, info = standardize_de_table(state_top_raw, "malignant_state_top")
std_tables.append(info)
cond_overall_sig, info = standardize_de_table(cond_overall_sig_raw, "condition_overall_significant")
std_tables.append(info)
cond_overall_top, info = standardize_de_table(cond_overall_top_raw, "condition_overall_top")
std_tables.append(info)
within_state_top, info = standardize_de_table(within_state_top_raw, "within_state_condition_top")
std_tables.append(info)

std_info_df = pd.DataFrame(std_tables)
write_tsv_replace(std_info_df, METADATA_DIR / f"{OUT_PREFIX}_detected_DE_columns.tsv")
display(std_info_df)


In [ ]:

# -------------------------
# Build gene universe
# -------------------------

def read_gene_universe_from_all_genes_files():
    patterns = [
        f"condition_malignant_pseudobulk_DE_{TAG}_overall_malignant_pairwise_DE_all_genes*.tsv",
        f"condition_malignant_pseudobulk_DE_{TAG}_overall_malignant_pairwise_DE_all_genes*.tsv.gz",
        f"condition_malignant_pseudobulk_DE_{TAG}_within_state_condition_DE_all_genes*.tsv",
        f"condition_malignant_pseudobulk_DE_{TAG}_within_state_condition_DE_all_genes*.tsv.gz",
        f"malignant_state_pseudobulk_DE_state_vs_rest_all_genes*.tsv",
        f"malignant_state_pseudobulk_DE_state_vs_rest_all_genes*.tsv.gz",
    ]
    files = []
    for pat in patterns:
        files.extend(sorted(METADATA_DIR.glob(pat)))
    genes = set()
    used = []
    for fp in files:
        try:
            if str(fp).endswith(".gz"):
                df = pd.read_csv(fp, sep="\t", compression="gzip", nrows=None, low_memory=False)
            else:
                df = pd.read_csv(fp, sep="\t", low_memory=False)
            gc = infer_gene_col(df)
            if gc is not None:
                genes.update(df[gc].dropna().astype(str).str.upper().tolist())
                used.append(str(fp))
        except Exception as e:
            print("[WARNING] Could not read universe file:", fp, repr(e))
    return genes, used

universe_genes, universe_files = read_gene_universe_from_all_genes_files()

fallback_genes = set()
for d in [state_sig, state_top, cond_overall_sig, cond_overall_top, within_state_top]:
    if d is not None and not d.empty:
        fallback_genes.update(d["gene_std"].dropna().astype(str).str.upper().tolist())

if len(universe_genes) < 1000:
    universe_source = "fallback_unique_genes_from_loaded_DE_tables"
    universe_genes = fallback_genes
else:
    universe_source = "all_genes_DE_files"

universe_report = pd.DataFrame([{
    "universe_source": universe_source,
    "n_universe_genes": len(universe_genes),
    "n_universe_files": len(universe_files),
    "universe_files": ";".join(universe_files[:20]),
    "warning": "Fallback universe may inflate ORA; prefer all_genes DE files if available." if universe_source.startswith("fallback") else ""
}])
write_tsv_replace(universe_report, METADATA_DIR / f"{OUT_PREFIX}_gene_universe_report.tsv")
display(universe_report)


In [ ]:

# -------------------------
# Curated glioma/scRNA-relevant gene sets
# -------------------------

curated_gene_sets = {
    "OPC_PRONEURAL": [
        "OLIG1","OLIG2","PDGFRA","SOX10","CSPG4","PTPRZ1","BCAN","DLL3","ASCL1","PLP1","PLLP","SHISA4","NKX2-2"
    ],
    "NPC_NEURAL": [
        "SOX11","DCX","RBFOX2","ELAVL4","TUBB3","STMN2","NHLH2","SCRT2","SLC17A6","NEUROD1","NEUROD6"
    ],
    "AC_REACTIVE_ASTROCYTIC": [
        "GFAP","AQP4","ALDH1L1","CLU","ID3","ID4","SLC1A3","APOE","VIM","FAM107A","S100B","SPARCL1"
    ],
    "HYPOXIA_STRESS": [
        "NDRG1","ADM","EGLN3","P4HA1","VEGFA","SLC2A1","HILPDA","PGK1","LDHA","ENO1","BNIP3","CA9","DDIT4","ALDOA"
    ],
    "GLYCOLYSIS_METABOLIC_STRESS": [
        "SLC2A1","HK2","PFKP","ALDOA","GAPDH","PGK1","ENO1","PKM","LDHA","PDK1","TPI1"
    ],
    "MES_ECM_REACTIVE": [
        "CHI3L1","CD44","VIM","FN1","COL1A1","COL1A2","COL3A1","TNC","SERPINE1","TIMP1","TAGLN","ANXA1","LGALS3","CCL2"
    ],
    "CYCLING_G2M_MITOTIC": [
        "MKI67","TOP2A","UBE2C","CENPF","PBK","ASPM","RRM2","HMGB2","BIRC5","DLGAP5","NUSAP1","CCNB1","CCNB2","CDC20"
    ],
    "E2F_S_PHASE": [
        "MCM2","MCM3","MCM4","MCM5","MCM6","MCM7","PCNA","TYMS","RRM1","RRM2","E2F1","E2F2","CDC6","CDK2"
    ],
    "MYELOID_INFLAMMATORY": [
        "LYZ","C1QA","C1QB","C1QC","APOE","TREM2","FCN1","SPP1","AIF1","CD68","TYROBP","LST1","MS4A7","HLA-DRA","HLA-DRB1","IL1B"
    ],
    "T_NK_CYTOTOXIC": [
        "CD3D","CD3E","TRAC","TRBC1","NKG7","GNLY","GZMB","GZMH","PRF1","KLRD1","IL7R","CD8A","CD4"
    ],
    "B_PLASMA": [
        "CD79A","CD79B","MS4A1","MZB1","JCHAIN","IGHG1","IGKC","SDC1","XBP1","BANK1"
    ],
    "ENDOTHELIAL_ANGIOGENESIS": [
        "PECAM1","VWF","KDR","CLDN5","FLT1","ESAM","RAMP2","EMCN","ANGPT2","PLVAP","CDH5"
    ],
    "PERICYTE_FIBROBLAST_ECM": [
        "PDGFRB","RGS5","ACTA2","TAGLN","DCN","LUM","COL1A1","COL1A2","COL3A1","COL6A1","COL6A2","MMP2"
    ],
    "INTERFERON_RESPONSE": [
        "ISG15","IFIT1","IFIT2","IFIT3","MX1","MX2","OAS1","OAS2","STAT1","IRF7","IFI6","IFI44L","CXCL10"
    ],
    "TNFA_NFKB_INFLAMMATION": [
        "NFKBIA","TNFAIP3","RELB","IL6","IL1B","CXCL8","CCL2","CCL3","CCL4","ICAM1","JUNB","FOS"
    ],
    "DNA_REPAIR_DAMAGE_RESPONSE": [
        "BRCA1","BRCA2","RAD51","MRE11","ATM","ATR","CHEK1","CHEK2","TP53","GADD45A","DDB2","XPC"
    ],
}

gene_sets = clean_gene_set_dict(curated_gene_sets)
external_gmts = collect_external_gmts()
external_loaded = []

for gmt in external_gmts:
    try:
        parsed = parse_gmt(gmt)
        parsed = clean_gene_set_dict(parsed)
        # Prefix external set names with file stem to avoid collisions.
        for name, genes in parsed.items():
            gene_sets[f"{gmt.stem}::{name}"] = genes
        external_loaded.append({"gmt_file": str(gmt), "n_gene_sets": len(parsed)})
    except Exception as e:
        external_loaded.append({"gmt_file": str(gmt), "n_gene_sets": 0, "error": repr(e)})

gene_set_inventory = pd.DataFrame([
    {
        "gene_set": name,
        "n_genes_raw": len(genes),
        "n_genes_in_universe": len(set(genes) & set(universe_genes)),
        "source": "external_gmt" if "::" in name else "curated_internal"
    }
    for name, genes in gene_sets.items()
]).sort_values(["source", "gene_set"]).reset_index(drop=True)

external_loaded_df = pd.DataFrame(external_loaded)
write_tsv_replace(gene_set_inventory, METADATA_DIR / f"{OUT_PREFIX}_gene_set_inventory.tsv")
write_tsv_replace(external_loaded_df, METADATA_DIR / f"{OUT_PREFIX}_external_gmt_files_loaded.tsv")
display(gene_set_inventory.head(30))
display(external_loaded_df)


In [ ]:

# -------------------------
# Build gene lists and run ORA
# -------------------------

state_lists = make_gene_lists_from_de(state_sig if len(state_sig) > 0 else state_top, "state_marker", q_cut=0.05, abs_lfc_cut=0.5)
condition_overall_lists = make_gene_lists_from_de(cond_overall_sig if len(cond_overall_sig) > 0 else cond_overall_top, "condition_overall", q_cut=0.05, abs_lfc_cut=0.5)
within_state_lists = make_gene_lists_from_de(within_state_top, "within_state_condition", q_cut=0.05, abs_lfc_cut=0.5)

gene_list_inventory = pd.DataFrame([
    {"analysis": "state_marker", "gene_list": k, "n_genes": len(v)} for k, v in state_lists.items()
] + [
    {"analysis": "condition_overall", "gene_list": k, "n_genes": len(v)} for k, v in condition_overall_lists.items()
] + [
    {"analysis": "within_state_condition", "gene_list": k, "n_genes": len(v)} for k, v in within_state_lists.items()
]).sort_values(["analysis", "gene_list"]).reset_index(drop=True)

write_tsv_replace(gene_list_inventory, METADATA_DIR / f"{OUT_PREFIX}_gene_list_inventory.tsv")
display(gene_list_inventory.head(40))

state_enrich = run_ora(state_lists, gene_sets, universe_genes, min_overlap=2)
condition_overall_enrich = run_ora(condition_overall_lists, gene_sets, universe_genes, min_overlap=2)
within_state_enrich = run_ora(within_state_lists, gene_sets, universe_genes, min_overlap=2)

write_tsv_replace(state_enrich, METADATA_DIR / f"{OUT_PREFIX}_malignant_state_marker_ORA.tsv")
write_tsv_replace(condition_overall_enrich, METADATA_DIR / f"{OUT_PREFIX}_condition_overall_malignant_ORA.tsv")
write_tsv_replace(within_state_enrich, METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_ORA.tsv")

display(state_enrich.head(30))
display(condition_overall_enrich.head(30))
display(within_state_enrich.head(30))


In [ ]:

# -------------------------
# Summarize strongest pathway interpretations
# -------------------------

def summarize_top_enrichment(enrich_df, analysis_label, top_n_per_list=5, q_cut=0.10):
    if enrich_df is None or enrich_df.empty:
        return pd.DataFrame()
    rows = []
    for gl, sub in enrich_df.groupby("gene_list"):
        sub = sub.sort_values(["q_value", "p_value", "odds_ratio"], ascending=[True, True, False])
        sub = sub[sub["q_value"] <= q_cut].head(top_n_per_list)
        for _, r in sub.iterrows():
            rows.append({
                "analysis": analysis_label,
                "gene_list": gl,
                "gene_set": r["gene_set"],
                "q_value": r["q_value"],
                "odds_ratio": r["odds_ratio"],
                "overlap_n": r["overlap_n"],
                "selected_n": r["selected_n"],
                "overlap_genes": r["overlap_genes"]
            })
    return pd.DataFrame(rows)

interpretation_summary = pd.concat([
    summarize_top_enrichment(state_enrich, "state_marker", top_n_per_list=5, q_cut=0.10),
    summarize_top_enrichment(condition_overall_enrich, "condition_overall", top_n_per_list=5, q_cut=0.10),
    summarize_top_enrichment(within_state_enrich, "within_state_condition", top_n_per_list=3, q_cut=0.10),
], ignore_index=True)

write_tsv_replace(interpretation_summary, METADATA_DIR / f"{OUT_PREFIX}_biological_interpretation_summary.tsv")
display(interpretation_summary.head(80))


In [ ]:

# -------------------------
# Program-score summary tables
# -------------------------

program_global_out = program_global.copy()
program_pairwise_out = program_pairwise.copy()

if not program_global_out.empty:
    write_tsv_replace(program_global_out, METADATA_DIR / f"{OUT_PREFIX}_program_score_global_kruskal_summary.tsv")
    display(program_global_out.head(30))

if not program_pairwise_out.empty:
    write_tsv_replace(program_pairwise_out, METADATA_DIR / f"{OUT_PREFIX}_program_score_pairwise_summary.tsv")
    display(program_pairwise_out.head(50))


In [ ]:

# -------------------------
# Figures
# -------------------------

figure_records = []

p = plot_enrichment_heatmap(
    state_enrich,
    FIG_DIR / f"{OUT_PREFIX}_state_marker_ORA_heatmap.png",
    "Malignant-state marker enrichment",
    top_gene_sets=25,
    top_lists=20
)
if p is not None:
    figure_records.append({"figure": "state_marker_ORA_heatmap", "path": str(p)})

p = plot_enrichment_heatmap(
    condition_overall_enrich,
    FIG_DIR / f"{OUT_PREFIX}_condition_overall_ORA_heatmap.png",
    "Condition-specific malignant pseudobulk enrichment",
    top_gene_sets=25,
    top_lists=30
)
if p is not None:
    figure_records.append({"figure": "condition_overall_ORA_heatmap", "path": str(p)})

p = plot_enrichment_heatmap(
    within_state_enrich,
    FIG_DIR / f"{OUT_PREFIX}_within_state_condition_ORA_heatmap.png",
    "Within-state condition-specific enrichment",
    top_gene_sets=25,
    top_lists=35
)
if p is not None:
    figure_records.append({"figure": "within_state_condition_ORA_heatmap", "path": str(p)})

# Program score global q-value barplot if columns can be inferred
if program_global is not None and not program_global.empty:
    prog_col = first_existing_col(program_global, ["program", "gene_program", "score", "module"])
    q_col = infer_q_col(program_global)
    p_col = infer_p_col(program_global)
    if prog_col is not None and (q_col is not None or p_col is not None):
        d = program_global.copy()
        val_col = q_col if q_col is not None else p_col
        d["val"] = pd.to_numeric(d[val_col], errors="coerce")
        d = d.dropna(subset=["val"]).sort_values("val")
        d["neglog10"] = -np.log10(np.clip(d["val"].values, 1e-300, 1.0))
        plt.figure(figsize=(8, max(4, 0.35 * len(d) + 1)))
        plt.barh(range(len(d)), d["neglog10"].values)
        plt.yticks(range(len(d)), d[prog_col].astype(str).tolist())
        plt.xlabel(f"-log10({val_col})")
        plt.title("Condition-associated malignant program score differences")
        plt.gca().invert_yaxis()
        out = FIG_DIR / f"{OUT_PREFIX}_program_score_global_barplot.png"
        plt.tight_layout()
        plt.savefig(out, dpi=200, bbox_inches="tight")
        plt.close()
        figure_records.append({"figure": "program_score_global_barplot", "path": str(out)})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(figure_manifest, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")
display(figure_manifest)


In [ ]:

# -------------------------
# Output manifest
# -------------------------

outputs = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_detected_DE_columns.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gene_universe_report.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gene_set_inventory.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_external_gmt_files_loaded.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gene_list_inventory.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_malignant_state_marker_ORA.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_condition_overall_malignant_ORA.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_ORA.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_interpretation_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_program_score_global_kruskal_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_program_score_pairwise_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
]

manifest = []
for p in outputs:
    p = Path(p)
    manifest.append({
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan
    })
manifest_df = pd.DataFrame(manifest)
write_tsv_replace(manifest_df, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest_df)
print("[DONE] Functional enrichment/pathway interpretation notebook completed.")
